## Q2 — DNF Handling
Drivers who did not finish (DNF) are retained in the dataset because `results.csv` 
assigns them a `positionOrder` ranked after all finishers (e.g., a DNF driver in a 
20-car race gets positionOrder 15–20). We made a deliberate choice to include DNFs 
rather than filter them out, as their pit stop data is still valid and meaningful. 
However, their finishing position reflects retirement order, not race performance.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

In [0]:
df_pit_stops = spark.read.csv("/Volumes/gr5069/raw/f1_data/pit_stops.csv", header=True)
df_races     = spark.read.csv("/Volumes/gr5069/raw/f1_data/races.csv",     header=True)
df_results   = spark.read.csv("/Volumes/gr5069/raw/f1_data/results.csv",   header=True)

In [0]:
#Parse duration → seconds
df_pit_stops = df_pit_stops.withColumn(
    "duration_sec",
    F.when(
        F.col("duration").contains(":"),
        F.split(F.col("duration"), ":").getItem(0).cast("double") * 60 +
        F.split(F.col("duration"), ":").getItem(1).cast("double")
    ).otherwise(
        F.col("duration").cast("double")
    )
)

In [0]:

# Avg pit stop time per driver per race
df_driver_avg = (
    df_pit_stops
    .groupBy("raceId", "driverId")
    .agg(F.round(F.avg("duration_sec"), 3).alias("avg_pit_stop_sec"))
)

In [0]:
#Finished Postion
df_results = df_results.select(
    "raceId", "driverId",
    F.col("positionOrder").cast("int").alias("finishing_position")
)

In [0]:
df_joined = (
    df_driver_avg
    .join(df_results, on=["raceId", "driverId"], how="left")
    .join(df_races.select("raceId", "year", "name"), on="raceId", how="left")
)

In [0]:
#Using Window Function to rank
window = Window.partitionBy("raceId").orderBy("finishing_position")

df_q2 = (
    df_joined
    .withColumn("rank", F.rank().over(window))
    .select("year", "name", "finishing_position", "rank", "driverId", "avg_pit_stop_sec")
    .orderBy("year", "name", "finishing_position")
)


In [0]:
display(df_q2)